#Exercícios Complementares

Carregue o dataset a seguir.

In [ ]:
import requests, os


filename = 'games.csv'

def download (filename):
  url = f'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/{filename}'

  if (os.path.isfile(f'./{filename}')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return

  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(filename)



Download realizado e arquivo extraído no Runtime... Tudo OK


Esse exercício é similar ao da última aula. No entanto, a idéia é identificar o classificador que gera a melhor ROC-AUC, mas utilizando os recursos de lematização, remoção de stopwords e ngramas de 1-3.

Esse dataset contém 3 colunas: title-abstract, title e label. Implemente as seguintes operações:
1. Extraia o corpus da coluna abstract e a classe dos documentos da coluna label;
2. (**Novo**) Remova as stopwords e execute a lematização para os termos. Observem que os documentos estão escritos em inglês;
3. Separe os dados em um grupo de treinamento e teste com o método train_test_split, com test_size de 0.3 e random_state de 42;
4. (**Novo**) Configure a extração de características com o TF-IDF, com ngramas de 1-3;
5. Considerando os classificadores de Árvores de Decisão, Random Forest, Linear SVM e Regressão Logística; identifique qual dos classificadores gera a melhor acurácia de acordo com a métrica de ROC-AUC.


**Resposta:** Linear SVM com ROC-AUC = 0.615

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

import pandas as pd, spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


df = pd.read_csv(filename)
corpus = df['title-abstract'].tolist()
y = df['label'].tolist()

nlp = spacy.load('en_core_web_sm')

for i in range(len(corpus)):
  doc = nlp(corpus[i])
  corpus[i] = ' '.join([token.lemma_ for token in doc if not token.is_stop])

X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size=0.3, random_state=42)

vectorizer = TfidfVectorizer(ngram_range=(1, 3))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

classifiers = [
  DecisionTreeClassifier(),
  RandomForestClassifier(),
  LinearSVC(),
  LogisticRegression()
]

for classifier in classifiers:
  classifier.fit(X_train_vectorized, y_train)
  y_pred = classifier.predict(X_test_vectorized)
  roc_auc = roc_auc_score(y_test, y_pred)
  print(f'{classifier.__class__.__name__}: ROC-AUC = {roc_auc}')
  print(classification_report(y_test, y_pred))
  print(confusion_matrix(y_test, y_pred))



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
DecisionTreeClassifier: ROC-AUC = 0.5683139534883721
              precision    recall  f1-score   support

           0       0.71      0.51      0.59        43
           1       0.42      0.62      0.50        24

    accuracy                           0.55        67
   macro avg       0.56      0.57      0.55        67
weighted avg       0.60      0.55      0.56        67

[[22 21]
 [ 9 15]]
RandomForestClassifier: ROC-AUC = 0.5184108527131783
              precision    recall  f1-score   support

           0       0.65      0.95      0.77        43
           1       0.5

/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LogisticRegression: ROC-AUC = 0.5067829457364341
              precision    recall  f1-score   support

           0       0.65      0.93      0.76        43
           1       0.40      0.08      0.14        24

    accuracy                           0.63        67
   macro avg       0.52      0.51      0.45        67
weighted avg       0.56      0.63      0.54        67

[[40  3]
 [22  2]]


Utilizem outros recursos aprendidos na disciplina de Aprendizado de Máquina para tentar atingir uma ROC-AUC maior com esses classificadores.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import Pipeline


corpus = df['title-abstract'].tolist()
y = df['label'].tolist()

for i in range(len(corpus)):
  doc = nlp(corpus[i])
  corpus[i] = ' '.join([token.lemma_ for token in doc if not token.is_stop])

X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size=0.3, random_state=42)

config = {
  'Decision Tree': {
    'model': DecisionTreeClassifier(random_state=42),
    'params': {
      'classifier__max_depth': [None, 10, 20],
      'classifier__min_samples_split': [2, 5, 10],
      'classifier__min_samples_leaf': [1, 2, 4],
      'selector__k': [100, 500, 1000]
    }
  },
  'Random Forest': {
    'model': RandomForestClassifier(random_state=42),
    'params': {
      'classifier__n_estimators': [10, 50, 100],
      'classifier__max_depth': [None, 10, 20],
      'classifier__min_samples_split': [2, 5, 10],
      'classifier__min_samples_leaf': [1, 2, 4],
      'selector__k': [100, 500, 1000]
    }
  },
  'Linear SVM': {
    'model': LinearSVC(random_state=42),
    'params': {
      'classifier__C': [1, 10, 100],
      'classifier__tol': [1e-3, 1e-4, 1e-5],
      'selector__k': [100, 500, 1000]
    }
  },
  'Logistic Regression': {
    'model': LogisticRegression(random_state=42),
    'params': {
      'classifier__C': [1, 10, 100],
      'classifier__tol': [1e-3, 1e-4, 1e-5],
      'selector__k': [100, 500, 1000]
    }
  }
}

models = []
for name, cfg in config.items():
  models.append((name, GridSearchCV(Pipeline(
        [('extractor', TfidfVectorizer(ngram_range=(1,3))),
        ('selector', SelectKBest()),
        ('classifier', cfg['model'])]
  ), cfg['params'], cv=2)))


for (name, model) in models:
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  roc_auc = roc_auc_score(y_test, y_pred)
  print(f'{name}: ROC-AUC = {roc_auc}')
  print(classification_report(y_test, y_pred))
  print(confusion_matrix(y_test, y_pred))



Decision Tree: ROC-AUC = 0.5847868217054263
              precision    recall  f1-score   support

           0       0.71      0.63      0.67        43
           1       0.45      0.54      0.49        24

    accuracy                           0.60        67
   macro avg       0.58      0.58      0.58        67
weighted avg       0.62      0.60      0.60        67

[[27 16]
 [11 13]]
Random Forest: ROC-AUC = 0.45542635658914726
              precision    recall  f1-score   support

           0       0.62      0.74      0.67        43
           1       0.27      0.17      0.21        24

    accuracy                           0.54        67
   macro avg       0.44      0.46      0.44        67
weighted avg       0.49      0.54      0.51        67

[[32 11]
 [20  4]]


/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_classes.py:32: Futu

Linear SVM: ROC-AUC = 0.6361434108527131
              precision    recall  f1-score   support

           0       0.73      0.81      0.77        43
           1       0.58      0.46      0.51        24

    accuracy                           0.69        67
   macro avg       0.65      0.64      0.64        67
weighted avg       0.68      0.69      0.68        67

[[35  8]
 [13 11]]
Logistic Regression: ROC-AUC = 0.4859496124031008
              precision    recall  f1-score   support

           0       0.63      0.93      0.75        43
           1       0.25      0.04      0.07        24

    accuracy                           0.61        67
   macro avg       0.44      0.49      0.41        67
weighted avg       0.50      0.61      0.51        67

[[40  3]
 [23  1]]
